In [8]:
# STEP 1: TRAIN & LOAD SENTENCEPIECE TOKENIZER

import os
import torch
import torch.nn as nn
from torch.nn import functional as F
import sentencepiece as spm

torch.manual_seed(66)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

VOCAB_SIZE = 10000
corpus_file = 'rap_corpus.txt'

print("Training SentencePiece tokenizer...")
spm.SentencePieceTrainer.train(
    input=corpus_file,
    input_format="text",
    model_prefix="tokrap",
    model_type="bpe",
    vocab_size=VOCAB_SIZE,
    normalization_rule_name="identity",
    remove_extra_whitespaces=False,
    allow_whitespace_only_pieces=True,
    add_dummy_prefix=True,
    split_digits=False,
    character_coverage=1.0,
    byte_fallback=True,
    unk_id=0,
    bos_id=1,
    eos_id=2,
    pad_id=3,
    num_threads=os.cpu_count()
)
print("tokenizer training complete!")
sp = spm.SentencePieceProcessor(model_file='tokrap.model')

voc_size = sp.get_piece_size()
encode = lambda s: sp.encode(s, out_type=int)
decode = lambda l: sp.decode(l)

print(f"vocabulary size: {voc_size}")

Training SentencePiece tokenizer...
tokenizer training complete!
vocabulary size: 10000


In [10]:
# STEP 2: HYPERPARAMETERS & DATA PREPARATION

batch_size = 32
block_size = 128
n_emb = 384
n_head = 6
dropout_rate = 0.3
n_layer = 4
eval_interval = 500
eval_iters = 66
max_steps = 3000
learning_rate = 3e-4

with open(corpus_file, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"characters in dataset: {len(text):,}")

data = torch.tensor(encode(text), dtype=torch.long)
print(f"total tokens: {len(data):,}")

n = int(0.9 * len(data))
train_data = data[:n]
test_data = data[n:]

def get_batch(split):
    data_split = train_data if split == 'train' else test_data
    ix = torch.randint(len(data_split) - block_size, (batch_size,))
    x = torch.stack([data_split[i : i + block_size] for i in ix])
    y = torch.stack([data_split[i + 1 : i + block_size + 1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'test']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

characters in dataset: 2,521,083
total tokens: 771,757


In [11]:
# STEP 3: TRANSFORMER ARCHITECTURE

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_emb, head_size, bias=False)
        self.query = nn.Linear(n_emb, head_size, bias=False)
        self.value = nn.Linear(n_emb, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_emb)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    def __init__(self, n_emb):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_emb, 4 * n_emb),
            nn.ReLU(),
            nn.Linear(4 * n_emb, n_emb),
            nn.Dropout(dropout_rate)
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_emb, n_head):
        super().__init__()
        head_size = n_emb // n_head
        self.sa_head = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_emb)
        self.ln1 = nn.LayerNorm(n_emb)
        self.ln2 = nn.LayerNorm(n_emb)

    def forward(self, x):
        x = x + self.sa_head(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class RapformerLangModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(voc_size, n_emb)
        self.position_embedding_table = nn.Embedding(block_size, n_emb)
        self.blocks = nn.Sequential(*[Block(n_emb, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_emb)
        self.lm_head = nn.Linear(n_emb, voc_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx_trunc = idx[:, -block_size:]
            logits, _ = self(idx_trunc)
            logits = logits[:, -1, :]
            logits = logits / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = RapformerLangModel()
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

total_params = sum(p.numel() for p in model.parameters())
print(f"rapformer initialized on: {device}")
print(f"total parameters: {total_params:,}")

rapformer initialized on: cuda
total parameters: 14,833,168


In [12]:
# STEP 4: TRAINING LOOP

for step in range(max_steps):
    if step % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {step}: train loss {losses['train']:.4f}, test loss {losses['test']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

torch.save(model.state_dict(), 'rapformer_v0.pth')
print("model saved as 'rapformer_v0.pth'!")

step 0: train loss 9.3485, test loss 9.3518
step 500: train loss 4.6357, test loss 4.4387
step 1000: train loss 4.1805, test loss 4.1837
step 1500: train loss 3.8579, test loss 4.0650
step 2000: train loss 3.5827, test loss 4.0005
step 2500: train loss 3.3981, test loss 4.0120
model saved as 'rapformer_v0.pth'!


In [16]:
# STEP 5: GENERATION
print("\n--- rapformer awaken ... ---\n")

start_phrase = "This one's from zwix!\n"
context_tokens = encode(start_phrase)
context = torch.tensor([context_tokens], dtype=torch.long, device=device)

# coherent verse generation
generated_tokens = model.generate(context, max_new_tokens=600, temperature=1.2, top_k=50)[0].tolist()
print(decode(generated_tokens))


--- rapformer awaken ... ---

This one's from zwix!
You see jigaboos, we tried, took up in
Maybach. West Side
Pour-ass scheme out, let's be up

I can't let it show some up and give everything a hundred niggas still good
Promise the wave from them
I'ma need them bills
Flexin' on the rise
Touch Eem 'Ville on the wrist in their neck line (
I know the truth
I gotta make it look the loot, drink (
The way I'm down
Get a lick on my ride)
A lot of a new shit that I can't afford (
Catch
)
Fuck your shit, nah? You run away from your pussy niggas
Never let your life, gotta, that's gotta go up on your way because them bitches go
The streets ain't ain't too good, 'bout it sound like the one shot?
Ball gotta move, y'all got that
Tefchistry Pace
I just wanted you gotta check a fan, come around you to show every time
I ain't scared, you better call, not knowin' we gotta get laid outside in the wind
You tryna know that you ain't gotta catchph, let 'em in our own it
Sauce in on your way before you know